# 教師あり学習 ― 分類タスク
ワインのさまざまなパラメータから，原料となったブドウ品種を推定するAI (分類器) を，教師あり学習によって作成する

## データセットの読み込み

In [2]:
# scikit-learn (sklearn) ライブラリから，load_wine モジュールをインポートする
from sklearn.datasets import load_wine
# ワインデータセットを変数 wine に代入する
wine = load_wine()
# wine.data          : ndarray   : 説明変数（独立変数）
# wine.feature_names : list      : 変数名のリスト
# wine.target        : ndarray   : 目的変数（従属変数）
# wine.DESCR         : string    : データセットの概要

# pandas ライブラリを pd という名前でインポートする
import pandas as pd
# ワインデータセットの説明変数を，pandas のデータフレーム(表)の形式に変換して変数 df に代入する
df = pd.DataFrame(wine.data, columns = wine.feature_names)

# ワインデータセットの目的変数(ぶどうの栽培品種)を class という名前の列で df に追加する
df["class"] = wine.target

# 結果のデータフレームを表示（各列の意味はテキストp.220の表14-1を参照）
display(df)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [3]:
# df の各列の基本統計量を表示
display(df.describe())

# 欠損値がない (count がすべて行数と一致) ことを確認しておく

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000


## 訓練データとテストデータの分割

In [4]:
x = df.drop(["class"], axis=1)  # 説明変数 (class以外の列からなるデータフレーム)
y = df["class"]                 # 目的変数 (class列のみからなるシリーズ (データフレームの1列分))

# データをランダムに分割するためのライブラリ train_test_split をインポートする
from sklearn.model_selection import train_test_split

# 元データを訓練データとテストデータに分割する
#   train_size：    訓練データの割合または個数 (この例では70%)
#   test_size：     テストデータの割合または個数 (この例では30%)
#   random_state：  乱数の seed (種子)— 0以上4294967295以下の任意の整数 (同じ値に対しては，毎回同じように分割される)
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, test_size=0.3, random_state=0)

# 結果的に以下のように分割される (変数ビュー (上の[変数]ボタンで開く) で確認してみよう)
#   x_train：   訓練データの説明変数 (y_train と1対1に対応する) — 178の約70%，124組のデータとなる
#   x_test：    テストデータの説明変数 (y_test と1対1に対応する) — 178の約30%，54組のデータとなる
#   y_train：   訓練データの目的変数 (x_train と1対1に対応する) — 178の約70%，124個のデータとなる
#   y_test：    テストデータの目的変数 (x_test と1対1に対応する) — 178の約30%，54個のデータとなる

# 各データのデータ数 (行数) を表示
print("x_train のデータ数 (行数):", len(x_train), ", x_test のデータ数 (行数):", len(x_test))
print("y_train のデータ数 (行数):", len(y_train), ", y_test のデータ数 (行数):", len(y_test))

x_train のデータ数 (行数): 124 , x_test のデータ数 (行数): 54
y_train のデータ数 (行数): 124 , y_test のデータ数 (行数): 54


## ランダムフォレストを用いた分類器の生成

In [5]:
# ランダムフォレストのモジュール RandomForestClassifier をインポートする
from sklearn.ensemble import RandomForestClassifier

# ランダムフォレストのモデルを生成して変数 rf_model に代入する
# デフォルトでは，100個の決定木が生成される (n_estimators = 100)
rf_model = RandomForestClassifier(random_state=0)
# rf_model を訓練データ x_train, y_train で訓練する (学習させる)
rf_model.fit(x_train, y_train)

# 学習済みの rf_model に，テストデータの説明変数を入力して目的変数を推定させて表示
rf_test = rf_model.predict(x_test)
# 目的変数の推定値を表示
print(rf_test)
# テストデータの目的変数 (正解) を表示
print(y_test.values)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1 0 2 1 2 0 2 2 0 2]
[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 1 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1 0 2 1 2 0 2 2 0 2]


### 混同行列

In [6]:
# 混同行列のモジュール confusion_matrix をインポートする
from sklearn.metrics import confusion_matrix
# 混同行列を生成して表示
print(confusion_matrix(y_test, rf_test))

[[19  0  0]
 [ 0 21  1]
 [ 0  0 13]]


- 行が `y_test` の 0, 1, 2 に対応
- 列が `rf_test` の 0, 1, 2 に対応

### 分類精度

In [7]:
# 訓練データでの正解率を表示
print("正解率(train): ", rf_model.score(x_train, y_train))
# テストデータでの正解率を表示
print("正解率(test):  ", rf_model.score(x_test, y_test))

正解率(train):  1.0
正解率(test):   0.9814814814814815


## FNNを用いた分類器の作成

In [8]:
# FNNのモジュール MLPClassifier をインポートする
from sklearn.neural_network import MLPClassifier

# FNNのモデルを生成して変数 mlp_model に代入する
# デフォルトではユニット数100の隠れ層1層となる (hidden_layer_sizes=(100,))
# デフォルトの学習反復回数では「最適化が収束していない」という
# メッセージが出るので，学習反復回数を1000回とする (max_iter=1000)
mlp_model = MLPClassifier(random_state=0, max_iter=1000)
# mlp_model を訓練データ x_train, y_train で訓練する (学習させる)
mlp_model.fit(x_train, y_train)

# 学習済みの mlp_model にテストデータの説明変数を入力し，目的変数を推定させて表示
mlp_test = mlp_model.predict(x_test)
# 目的変数の推定値を表示
print(mlp_test)
# テストデータの目的変数 (正解) を表示
print(y_test.values)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1 0 2 0 2 0 2 2 0 2]
[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 1 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1 0 2 1 2 0 2 2 0 2]


### 混同行列

In [9]:
# 混同行列のモジュール confusion_matrix をインポートする
from sklearn.metrics import confusion_matrix
# 混同行列を生成して表示
print(confusion_matrix(y_test, mlp_test))

[[19  0  0]
 [ 1 20  1]
 [ 0  0 13]]


### 分類精度

In [10]:
# 訓練データでの正解率を表示
print("正解率(train): ", mlp_model.score(x_train, y_train))
# テストデータでの正解率を表示
print("正解率(test):  ", mlp_model.score(x_test, y_test))

正解率(train):  1.0
正解率(test):   0.9629629629629629


## 試してみよう
- 学習データとテストデータの分割の際の，`train_test_split` の `random_state`
  の値を0以外 (正の整数値) に変えてみよう（学習データとテストデータの分け方が変わる）
- ランダムフォレストの決定木の数を変更してみよう（例えば10個にするなら
  `RandomForestClassifier` の引数に `n_estimators=10`
  を書き足して `RandomForestClassifier(random_state=0, n_estimators=10)` とする）
- FNNの隠れ層のユニット数や層の数を変更してみよう（例えば，90個1層にするなら
  `MLPClassifier` の引数に `hidden_layer_sizes=(90,)`
  と書き足す．90個，60個の2層にするなら `hidden_layer_sizes=(90,60)` を書き足す）



In [14]:
rf_model10 = RandomForestClassifier(random_state=0, n_estimators=10)
rf_model10.fit(x_train, y_train)

print("正解率(train): ", rf_model10.score(x_train, y_train))
print("正解率(test):  ", rf_model10.score(x_test, y_test))

正解率(train):  1.0
正解率(test):   1.0


In [15]:
rf_model1 = RandomForestClassifier(random_state=0, n_estimators=1)
rf_model1.fit(x_train, y_train)

print("正解率(train): ", rf_model1.score(x_train, y_train))
print("正解率(test):  ", rf_model1.score(x_test, y_test))

正解率(train):  0.9354838709677419
正解率(test):   0.8518518518518519


In [30]:
mlp_model50 = MLPClassifier(random_state=0, max_iter=1000, hidden_layer_sizes=(50,))
mlp_model50.fit(x_train, y_train)

print("正解率(train): ", mlp_model50.score(x_train, y_train))
print("正解率(test):  ", mlp_model50.score(x_test, y_test))

正解率(train):  0.1935483870967742
正解率(test):   0.2222222222222222


In [25]:
mlp_model400 = MLPClassifier(random_state=0, max_iter=1000, hidden_layer_sizes=(400,))
mlp_model400.fit(x_train, y_train)

print("正解率(train): ", mlp_model400.score(x_train, y_train))
print("正解率(test):  ", mlp_model400.score(x_test, y_test))

正解率(train):  0.9919354838709677
正解率(test):   0.9444444444444444


In [26]:
mlp_model200_400 = MLPClassifier(random_state=0, max_iter=200, hidden_layer_sizes=(400,))
mlp_model200_400.fit(x_train, y_train)

print("正解率(train): ", mlp_model200_400.score(x_train, y_train))
print("正解率(test):  ", mlp_model200_400.score(x_test, y_test))

正解率(train):  0.9596774193548387
正解率(test):   0.9814814814814815


c:\ProgramData\Miniconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:702: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [27]:
mlp_model200_100 = MLPClassifier(random_state=0, max_iter=200, hidden_layer_sizes=(100,))
mlp_model200_100.fit(x_train, y_train)

print("正解率(train): ", mlp_model200_100.score(x_train, y_train))
print("正解率(test):  ", mlp_model200_100.score(x_test, y_test))

正解率(train):  0.9112903225806451
正解率(test):   0.9259259259259259


c:\ProgramData\Miniconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:702: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
